In [ ]:
#CELULA 1: Imports, Variaveis de Ambiente, GCP Client e Configuracoes Globais

import os
import json
import requests
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import storage

# 1. Autenticação Local GCP
BASE_DIR = os.path.dirname(os.path.abspath(""))
KEY_PATH = os.path.join(BASE_DIR, "config", "gcp_key.json")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# 2. Instanciação do Client do Google Cloud Storage
client = storage.Client.from_service_account_json(KEY_PATH)

# 3. Parâmetros Globais com Data Piso (Padrão de Produção / Backfill Controlado)
SERIE_CDI = 12  # SGS 12 = CDI diário (% a.d.)
BUCKET_BRONZE = "cvm-landing-zone-bronze-dev"

HOJE = datetime.now().date()
D1_LIMITE = HOJE - timedelta(days=1)

# DATA PISO: O pipeline NUNCA buscará dados anteriores a esta data
DATA_PISO_BACKFILL = datetime.strptime("2026-01-01", "%Y-%m-%d").date()

bucket = client.bucket(BUCKET_BRONZE)

print(f"✔ Client GCP inicializado para BCB CDI!")
print(f"✔ Target Bronze Bucket: gs://{BUCKET_BRONZE}")
print(f"✔ Data Piso de Backfill: {DATA_PISO_BACKFILL.strftime('%d/%m/%Y')}")
print(f"✔ Data Teto (D-1): {D1_LIMITE.strftime('%d/%m/%Y')}")

In [ ]:
#CELULA 2: Ingestao Incremental Dinamica com Piso de Backfill e Particionamento por Ano

def obter_dados_cdi_existentes(gcs_path: str, ano_alvo: int) -> tuple[list, datetime]:
    """
    Inspeciona a particao do ano no GCS.
    Retorna os dados existentes e a data de inicio para a proxima consulta.
    """
    blob = bucket.blob(gcs_path)
    data_inicio_padrao = max(datetime(ano_alvo, 1, 1).date(), DATA_PISO_BACKFILL)
    
    if not blob.exists():
        return [], data_inicio_padrao
        
    try:
        conteudo = blob.download_as_string()
        dados = json.loads(conteudo)
        if not dados:
            return [], data_inicio_padrao
            
        df_temp = pd.DataFrame(dados)
        df_temp['dt_obj'] = pd.to_datetime(df_temp['data'], format='%d/%m/%Y').dt.date
        maior_data_gravada = df_temp['dt_obj'].max()
        
        # O novo inicio sera o dia seguinte ao maior registro no GCS
        proxima_data = maior_data_gravada + timedelta(days=1)
        return dados, proxima_data
    except Exception as e:
        print(f"⚠ Erro ao ler partição {gcs_path} ({e}). Recalculando a partir do piso...")
        return [], data_inicio_padrao

def executar_ingestao_bcb_cdi_com_piso():
    ano_inicio = DATA_PISO_BACKFILL.year
    ano_fim = D1_LIMITE.year
    anos_para_processar = list(range(ano_inicio, ano_fim + 1))
    
    print(f"\n=== INICIANDO VARREDURA INCREMENTAL DO CDI BCB ===")
    print(f"Janela de análise: Anos {anos_para_processar} | Piso: {DATA_PISO_BACKFILL.strftime('%d/%m/%Y')}")
    
    for ano in anos_para_processar:
        ano_str = str(ano)
        gcs_path = f"bcb/cdi/ano={ano_str}/cdi_diario_{ano_str}.json"
        
        # Define limite superior do ano corrente (D-1) ou do ano passado (31/12)
        if ano < D1_LIMITE.year:
            teto_ano = datetime(ano, 12, 31).date()
        else:
            teto_ano = D1_LIMITE
            
        dados_existentes, dt_inicio = obter_dados_cdi_existentes(gcs_path, ano)
        
        # Garante que a data inicial nunca seja menor que o piso global
        if dt_inicio < DATA_PISO_BACKFILL:
            dt_inicio = DATA_PISO_BACKFILL
            
        if dt_inicio > teto_ano:
            print(f"✨ [BCB_CDI] Partição `ano={ano_str}` 100% atualizada (Até {teto_ano.strftime('%d/%m/%Y')}). Pulando...")
            continue
            
        str_inicio = dt_inicio.strftime("%d/%m/%Y")
        str_teto = teto_ano.strftime("%d/%m/%Y")
        
        print(f"🚀 [BCB_CDI] Buscando lacuna na partição `ano={ano_str}`: {str_inicio} até {str_teto}...")
        url_bcb = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{SERIE_CDI}/dados?formato=json&dataInicial={str_inicio}&dataFinal={str_teto}"
        
        try:
            response = requests.get(url_bcb, timeout=30)
            if response.status_code == 200:
                novos_dados = response.json()
                if not novos_dados:
                    print(f"ℹ Nenhum novo registro retornado pela API para {str_inicio} a {str_teto}.")
                    continue
                    
                todos_dados = dados_existentes + novos_dados
                df_consolidado = pd.DataFrame(todos_dados).drop_duplicates(subset=['data'], keep='last')
                
                df_consolidado['dt_obj'] = pd.to_datetime(df_consolidado['data'], format='%d/%m/%Y')
                df_consolidado = df_consolidado.sort_values('dt_obj').drop(columns=['dt_obj'])
                
                json_final = df_consolidado.to_json(orient='records', indent=2)
                
                blob = bucket.blob(gcs_path)
                blob.upload_from_string(json_final, content_type="application/json")
                
                print(f"✔ [BCB_CDI] Sucesso! Partição `ano={ano_str}` atualizada no GCS.")
                print(f"   └─ Novas linhas adicionadas: {len(novos_dados)}")
                print(f"   └─ Total na partição: {len(df_consolidado)} registros")
            else:
                print(f"❌ Erro HTTP {response.status_code} na API do BCB para o intervalo {str_inicio} - {str_teto}")
        except Exception as e:
            print(f"❌ Exceção ao consultar API do BCB: {str(e)}")

# Executa o pipeline resiliente com janela dinâmica e piso configurável
executar_ingestao_bcb_cdi_com_piso()